In [ ]:
%cd ..

/home/bhchen/LearnKalmanGain


In [19]:
import os
import glob
from PIL import Image

def create_gif_robust(image_folder, file_pattern, gif_path, duration=100, max_value=None):
    """
    Function:
        Creates a GIF from images matching a pattern with a numerical wildcard.
        This version is robust and does not depend on any keywords like 'timestep'.
        It identifies the number by what the '*' in the pattern matches.
    Input:
        image_folder (str): The path to the folder containing the images.
        file_pattern (str): Filename pattern with one '*' as a wildcard for a number.
                            Example: "frame_*_render.png"
        gif_path (str): The path to save the output GIF file.
        duration (int): Duration (in milliseconds) for each frame.
        max_value (int, optional): The maximum value of the wildcard part to include.
                                If None, all matched images will be used.
    Output:
        None
    """
    if file_pattern.count('*') != 1:
        print("Error: The file_pattern must contain exactly one '*' wildcard.")
        return

    # Dynamically determine the prefix and suffix from the pattern
    prefix, suffix = file_pattern.split('*')
    
    search_path = os.path.join(image_folder, file_pattern)
    all_filenames = glob.glob(search_path)
    
    if not all_filenames:
        print(f"Error: No images found in '{image_folder}' matching '{file_pattern}'.")
        return

    files_with_values = []
    for f_path in all_filenames:
        basename = os.path.basename(f_path)
        # Extract the part of the filename that corresponds to the wildcard
        if basename.startswith(prefix) and basename.endswith(suffix):
            try:
                # Remove prefix and suffix to get the numerical part
                value_str = basename[len(prefix):-len(suffix)]
                value = int(value_str)
                files_with_values.append({'value': value, 'path': f_path})
            except (ValueError, IndexError):
                # Ignore files where the wildcard part is not a valid integer
                print(f"Warning: Could not extract a valid number from '{basename}'. Skipping.")
                continue

    if not files_with_values:
        print("Error: Found matching files, but could not extract numbers from any of them.")
        return

    # Sort the files based on the extracted numerical value
    files_with_values.sort(key=lambda item: item['value'])

    # Filter by max_value if provided
    if max_value is not None:
        print(f"Filtering frames to include values up to {max_value}.")
        files_with_values = [item for item in files_with_values if item['value'] <= max_value]

    if not files_with_values:
        print(f"Error: After filtering, no images remained with a value <= {max_value}.")
        return
        
    final_filenames = [item['path'] for item in files_with_values]
    print(f"Creating GIF with {len(final_filenames)} frames...")

    images = [Image.open(fn) for fn in final_filenames]
    images[0].save(
        gif_path,
        save_all=True,
        append_images=images[1:],
        duration=duration,
        loop=0
    )
    print(f"GIF saved successfully at: {gif_path}")

In [ ]:

# --- CONFIGURE YOUR SETUP HERE ---

image_directory = "save/pf_vis" 
traj_index = 1
pattern = f"sigma_y1.0_batch64_len500_pfN1000000_timestep*_42_{traj_index}_fixed.png"

frame_duration_ms = 75
max_timestep_to_include = 500 

output_gif_file = f"save/pf_vis/L63_traj{traj_index}_{frame_duration_ms}ms_{max_timestep_to_include}timesteps.gif"

# --- END OF CONFIGURATION ---

# Run the function with your settings
create_gif_robust(
    image_folder=image_directory,
    file_pattern=pattern,
    gif_path=output_gif_file,
    duration=frame_duration_ms,
    max_value=max_timestep_to_include 
    )

Filtering frames to include values up to 500.
Creating GIF with 499 frames...
GIF saved successfully at: save/pf_vis/L63_traj1_75ms_500timesteps.gif


In [22]:
image_directory = "save/pf_vis" 
pattern = f"sigma_y1.0_batch64_len500_pfN1000000_timestep250_42_*.png"

frame_duration_ms = 400
max_value = 64

output_gif_file = f"save/pf_vis/L63_zoomin_{frame_duration_ms}ms_{max_timestep_to_include}timesteps.gif"

# Run the function with your settings
create_gif_robust(
    image_folder=image_directory,
    file_pattern=pattern,
    gif_path=output_gif_file,
    duration=frame_duration_ms,
    max_value=max_value 
)

Filtering frames to include values up to 64.
Creating GIF with 64 frames...
GIF saved successfully at: save/pf_vis/L63_zoomin_400ms_500timesteps.gif


In [ ]:
import torch
import ot # POT (Python Optimal Transport) library
import time

def wasserstein_distance_pt(x, y, n_projections=100):
    """
    Calculates the W-2 distance between two point clouds using PyTorch.
    - If d=1, it uses the efficient sorting-based method.
      Cost is approx. O(M*log(M) + N*log(N)) for each item in the batch.
    - If d>1, it uses the Sliced-Wasserstein distance.
      Cost is approx. O(L * K*log(K)) for each item, where L is the number of
      projections and K is the total number of points (M+N).

    INPUT:
    - x: A torch.Tensor of shape (M, d) or (B, M, d).
    - y: A torch.Tensor of shape (N, d) or (B, N, d).
    - n_projections: The number of random projections for the Sliced method.

    OUTPUT:
    - dist: A tensor containing the W-2 distance(s).
            Shape is (B,) for batched input, or a scalar for unbatched input.
    """
    # --- 1. Input Validation and Preparation ---
    if x.device != y.device:
        raise ValueError("Input tensors must be on the same device")
    
    original_device = x.device
    is_unbatched = x.dim() == 2
    if is_unbatched:
        x = x.unsqueeze(0)
        y = y.unsqueeze(0)

    B, M, d = x.shape
    _B, N, _d = y.shape

    if B != _B or d != _d:
      raise ValueError(f"Shape mismatch: x is {x.shape}, y is {y.shape}")

    # --- 2. Dimension-specific routing ---
    if d == 1:
        x_1d = x.squeeze(-1)
        y_1d = y.squeeze(-1)
        
        # Move to CPU for ot.wasserstein_1d which may not support CUDA
        dist_list_1d = [ot.wasserstein_1d(x_b.to("cpu"), y_b.to("cpu"), p=2) for x_b, y_b in zip(x_1d, y_1d)]
        dist = torch.stack(dist_list_1d).to(original_device)

    else: # d > 1, use Sliced-Wasserstein
        # FIX: Loop through the batch, as ot.sliced_wasserstein_distance
        # does not support batching in the (B, N, d) format.
        dist_list_sliced = [
            ot.sliced_wasserstein_distance(x_b, y_b, n_projections=n_projections, p=2)
            for x_b, y_b in zip(x, y)
        ]
        dist = torch.stack(dist_list_sliced)

    # --- 3. Final Output Formatting ---
    if is_unbatched:
        return dist.squeeze(0)
    return dist


# --- Main execution block for demonstration ---
if __name__ == '__main__':
    # Set device to GPU if available, otherwise CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")

    # --- === Example 1: 3D Unbatched Tensors (Sliced Method) === ---
    print("--- 1. 3D Unbatched Tensors (Sliced Method) ---")
    M, N, d = 500, 600, 3
    x_unbatched = torch.randn(M, d, device=device)
    y_unbatched = torch.randn(N, d, device=device)
    
    start = time.time()
    dist_sliced = wasserstein_distance_pt(x_unbatched, y_unbatched, n_projections=100)
    print(f"Sliced W-2: {dist_sliced.item():.4f} (Time: {(time.time() - start)*1000:.2f} ms)\n")
    

    # --- === Example 2: 3D Batched Tensors (Sliced Method) === ---
    print("--- 2. 3D Batched Tensors (Sliced Method) ---")
    B, M, N, d = 16, 500, 600, 3
    x_batched = torch.randn(B, M, d, device=device)
    y_batched = torch.randn(B, N, d, device=device)
    
    start = time.time()
    dist_sliced_b = wasserstein_distance_pt(x_batched, y_batched, n_projections=100)
    print(f"Sliced W-2 (batched): First value is {dist_sliced_b[0].item():.4f} (Time: {(time.time() - start)*1000:.2f} ms)\n")


    # --- === Example 3: 1D Batched Tensors === ---
    print("--- 3. 1D Batched Tensors ---")
    B, M, N, d = 32, 1000, 1500, 1
    x_1d = torch.randn(B, M, d, device=device)
    y_1d = torch.randn(B, N, d, device=device)
    
    # The function will automatically use the efficient 1D method
    start = time.time()
    dist_1d = wasserstein_distance_pt(x_1d, y_1d)
    print(f"1D W-2 (batched): First value is {dist_1d[0].item():.4f} (Time: {(time.time() - start)*1000:.2f} ms)")

Using device: cuda

--- 1. 3D Unbatched Tensors (Sliced Method) ---
Sliced W-2: 0.1109 (Time: 5.88 ms)

--- 2. 3D Batched Tensors (Sliced Method) ---
Sliced W-2 (batched): First value is 0.1098 (Time: 157.93 ms)

--- 3. 1D Batched Tensors ---
1D W-2 (batched): First value is 0.0089 (Time: 41.95 ms)
